# Patent phrase matching: starter notebook

This notebook is the **minimum working pipeline** for the mini project described in [project_brief.md](project_brief.md). It fine-tunes a pretrained Transformer **cross-encoder** to predict how similar two patent phrases are, checks it on a validation split, and writes `submission.csv`.

It is a starting point, not a solution. The baseline comparison, the experiments, and the error analysis in the brief are your work.

**Running it**

- Kaggle notebook, **GPU T4 or P100**, attached inputs:
  1. the competition data, via **Add Input → Competitions → U.S. Patent Phrase to Phrase Matching**;
  2. the output of [download_weights.ipynb](download_weights.ipynb), the saved pretrained checkpoint.
- Turn **Internet off** before you submit, because the competition runs submitted notebooks offline. The notebook falls back to the Hugging Face Hub only when it cannot find a local checkpoint, which works only with internet on.
- Writes to `/kaggle/working/`: `submission.csv` and `val_predictions.csv`. A rerun overwrites both. No checkpoints are saved.
- Uses packages preinstalled on Kaggle (`torch`, `transformers`, `datasets`, `scikit-learn`, `pandas`).

The report prompts are intended to be completed from your own run outputs and interpretation. No wording can guarantee a particular AI-detector result; focus on accurate results and being able to explain the work.

## 1. Setup

In [ ]:
import glob
import os

import numpy as np
import pandas as pd
import torch
import transformers
from datasets import Dataset
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tokenizers.processors import TemplateProcessing
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments)

SEED = 42
transformers.set_seed(SEED)   # seeds Python, NumPy, and PyTorch; GPU results can still vary slightly between runs
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| transformers", transformers.__version__, "| device:", device)

## 2. Configuration

These are the settings you will change in your experiments. Change **one** setting at a time so you can tell what caused a difference.

In [ ]:
MODEL_ID = "microsoft/deberta-v3-small"   # Hub name; the local copy is found by its last path component
MAX_LEN = 64            # anchor + target + context fit easily within 64 tokens
EPOCHS = 3
LR = 2e-5
BATCH_SIZE = 32
VAL_FRACTION = 0.2      # share of *anchors* (not rows) held out for validation
OUT_DIR = "/kaggle/working"


def find_one(pattern):
    """Return the directory containing the first matching file under /kaggle/input."""
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    return os.path.dirname(hits[0]) if hits else None


DATA_DIR = find_one("us-patent-phrase-to-phrase-matching/train.csv") or find_one("train.csv")
assert DATA_DIR, "Competition data not found: add it with Add Input -> Competitions."
MODEL_PATH = find_one(f"{MODEL_ID.split('/')[-1]}/config.json") or MODEL_ID
print("data:", DATA_DIR)
print("model:", MODEL_PATH, "(local)" if MODEL_PATH != MODEL_ID else "(Hub download: needs internet)")

## 3. Data

Each row asks how similar a **target** phrase is to an **anchor** phrase, judged within a patent **context**. The context is a CPC (Cooperative Patent Classification) code such as `G02`, whose first letter is the section. Scores are 0, 0.25, 0.5, 0.75, or 1. The visible `test.csv` has only a few rows; when you submit, Kaggle reruns the notebook with the hidden test set in its place.

In [ ]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
print("train:", train.shape, "| test (visible):", test.shape)
print(train.score.value_counts().sort_index())
train.head()

## 4. Model input

The model reads the anchor as the first text segment, and the target plus a short context description as the second. The tokenizer joins them as `[CLS] anchor [SEP] target; context [SEP]`, so attention can compare the two phrases token by token: this is what makes it a cross-encoder. Here the context is just the CPC section name. How you represent context is one of the experiments you can run.

In [ ]:
CPC_SECTIONS = {
    "A": "human necessities", "B": "performing operations; transporting",
    "C": "chemistry; metallurgy", "D": "textiles; paper", "E": "fixed constructions",
    "F": "mechanical engineering; lighting; heating; weapons; blasting",
    "G": "physics", "H": "electricity",
}


def add_inputs(df):
    df = df.copy()
    df["text_a"] = df["anchor"]
    df["text_b"] = df["target"] + "; " + df["context"].str[0].map(CPC_SECTIONS)
    return df


train, test = add_inputs(train), add_inputs(test)
train[["text_a", "text_b", "score"]].head()

## 5. Validation split

The training set contains only about 730 distinct anchors, each paired with many targets. If the same anchor appeared in both the training and validation sets, the validation score would overestimate performance on unseen anchors. The split therefore holds out whole **anchors**.

Keep this split fixed (same `SEED` and `VAL_FRACTION`) for every experiment, so your comparisons are fair.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=VAL_FRACTION, random_state=SEED)
train_idx, val_idx = next(splitter.split(train, groups=train["anchor"]))
trn, val = train.iloc[train_idx].reset_index(drop=True), train.iloc[val_idx].reset_index(drop=True)
assert set(trn.anchor).isdisjoint(val.anchor)
print(f"train rows {len(trn)} | val rows {len(val)} | val anchors {val.anchor.nunique()}")

## 6. Tokenization

Padding is left to the data collator, which pads each batch only to its longest sequence. The printed example should start with `[CLS]` and have a `[SEP]` between the two segments.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
# transformers 5.0 builds this tokenizer without [CLS]/[SEP]; set the pair template explicitly
tokenizer.backend_tokenizer.post_processor = TemplateProcessing(
    single="[CLS] $A [SEP]", pair="[CLS] $A [SEP] $B:1 [SEP]:1",
    special_tokens=[("[CLS]", tokenizer.cls_token_id), ("[SEP]", tokenizer.sep_token_id)])


def to_dataset(df, with_labels=True):
    cols = ["text_a", "text_b"] + (["score"] if with_labels else [])
    ds = Dataset.from_pandas(df[cols].astype({"score": "float32"}) if with_labels else df[cols])  # MSE loss needs float32 labels
    ds = ds.map(lambda b: tokenizer(b["text_a"], b["text_b"], truncation=True, max_length=MAX_LEN),
                batched=True, remove_columns=["text_a", "text_b"])
    return ds.rename_column("score", "labels") if with_labels else ds


ds_trn, ds_val, ds_test = to_dataset(trn), to_dataset(val), to_dataset(test, with_labels=False)
print(tokenizer.decode(ds_trn[0]["input_ids"]))

## 7. Fine-tuning

With `num_labels=1`, the classification head outputs a single number. Because the labels are floats, Hugging Face treats the task as **regression** and minimizes mean squared error. The competition metric is the **Pearson correlation** between predictions and true scores. Pearson correlation ignores any linear rescaling of the predictions, so the model is rewarded for ranking pairs in the right order, not for hitting the exact values.

In [ ]:
def safe_pearson(y_true, y_pred):
    """Pearson correlation that returns NaN instead of raising on constant arrays."""
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    if len(y_true) < 2 or np.std(y_true) == 0 or np.std(y_pred) == 0:
        return float("nan")
    return float(np.corrcoef(y_true, y_pred)[0, 1])


def compute_metrics(eval_pred):
    preds, labels = eval_pred
    return {"pearson": safe_pearson(labels, preds)}


# Load in fp32: the deberta-v3 checkpoints are stored in fp16, and fp16=True mixed precision below needs
# fp32 weights ("Attempting to unscale FP16 gradients" otherwise). Older transformers call this torch_dtype.
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH, num_labels=1, dtype=torch.float32)
args = TrainingArguments(
    output_dir=f"{OUT_DIR}/trainer_tmp",
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=2 * BATCH_SIZE,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",          # keeps disk use and runtime low; the final weights stay in memory
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)
trainer = Trainer(model=model, args=args, train_dataset=ds_trn, eval_dataset=ds_val,
                  data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics)
trainer.train()

## 8. Validation predictions

The predictions are saved so you can do your error analysis without retraining.

In [ ]:
val["pred"] = trainer.predict(ds_val).predictions.reshape(-1)
val_pearson = np.corrcoef(val["pred"], val["score"])[0, 1]
print(f"Validation Pearson: {val_pearson:.4f}")
val[["id", "anchor", "target", "context", "score", "pred"]].to_csv(f"{OUT_DIR}/val_predictions.csv", index=False)

## 9. Baseline without fine-tuning

This baseline uses TF-IDF word and character features to compare the two phrases. It does not train a Transformer. The same anchor-grouped validation split is used, so its Pearson score can be compared with the fine-tuned model. This is a simple lexical baseline: it may miss semantic similarity when two phrases use different words.

In [ ]:
# Combine word-level and character-level TF-IDF features.
# Character n-grams help with spelling variants and technical word forms.
baseline_text = (trn["anchor"] + " " + trn["target"]).astype(str)
val_text = (val["anchor"] + " " + val["target"]).astype(str)

word_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2), min_df=1, sublinear_tf=True,
    strip_accents="unicode", token_pattern=r"(?u)\\b\\w+\\b"
)
char_vectorizer = TfidfVectorizer(
    analyzer="char", ngram_range=(2, 5), min_df=1,
    sublinear_tf=True, max_features=100_000
)

# Fit only on training text to avoid validation leakage.
word_vectorizer.fit(baseline_text)
char_vectorizer.fit(baseline_text)

def lexical_similarity(frame):
    a_word = word_vectorizer.transform(frame["anchor"].astype(str))
    b_word = word_vectorizer.transform(frame["target"].astype(str))
    a_char = char_vectorizer.transform(frame["anchor"].astype(str))
    b_char = char_vectorizer.transform(frame["target"].astype(str))
    word_sim = np.asarray(a_word.multiply(b_word).sum(axis=1)).ravel()
    char_sim = np.asarray(a_char.multiply(b_char).sum(axis=1)).ravel()
    return 0.5 * word_sim + 0.5 * char_sim

val["baseline_pred"] = lexical_similarity(val)
baseline_pearson = safe_pearson(val["score"], val["baseline_pred"])
print(f"TF-IDF baseline validation Pearson: {baseline_pearson:.4f}")

## 10. Controlled experiments

The starter model is the reference run. The two experiments below change one variable at a time:

- **Experiment A:** increase `MAX_LEN` while keeping the learning rate and other settings fixed.
- **Experiment B:** lower `LR` while keeping the sequence length and other settings fixed.

Each experiment starts from the same pretrained checkpoint and uses the same train/validation rows. Run them one at a time if GPU memory or notebook runtime is limited. The flag is off by default so the notebook can first produce the baseline submission without automatically launching two additional training jobs. Set it to `True` to run the experiments and record the actual scores.

In [ ]:
RUN_CONTROLLED_EXPERIMENTS = False  # Change to True to run both additional fine-tuning jobs.

def run_one_experiment(experiment_name, max_len, learning_rate):
    # Re-tokenize for the selected sequence length.
    original_max_len = MAX_LEN
    ds_train_exp = to_dataset(trn)
    ds_val_exp = to_dataset(val)

    # `to_dataset` reads the global MAX_LEN, so the caller sets it before this function.
    exp_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_PATH, num_labels=1, dtype=torch.float32
    )
    exp_args = TrainingArguments(
        output_dir=f"{OUT_DIR}/{experiment_name}",
        num_train_epochs=EPOCHS,
        learning_rate=learning_rate,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=2 * BATCH_SIZE,
        warmup_ratio=0.1,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=100,
        fp16=torch.cuda.is_available(),
        report_to="none",
        seed=SEED,
    )
    exp_trainer = Trainer(
        model=exp_model,
        args=exp_args,
        train_dataset=ds_train_exp,
        eval_dataset=ds_val_exp,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics,
    )
    exp_trainer.train()
    predictions = exp_trainer.predict(ds_val_exp).predictions.reshape(-1)
    score = safe_pearson(val["score"], predictions)
    print(f"{experiment_name}: max_len={max_len}, lr={learning_rate}, Pearson={score:.4f}")
    del exp_trainer, exp_model, exp_args, ds_train_exp, ds_val_exp
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {"experiment": experiment_name, "max_len": max_len,
            "learning_rate": learning_rate, "validation_pearson": score}


experiment_results = []
if RUN_CONTROLLED_EXPERIMENTS:
    # A: sequence length only changes; learning rate stays at the starter value.
    MAX_LEN = 96
    experiment_results.append(run_one_experiment("experiment_max_len_96", 96, LR))

    # B: learning rate only changes relative to the starter; restore sequence length.
    MAX_LEN = 64
    experiment_results.append(run_one_experiment("experiment_lr_1e5", 64, 1e-5))
    MAX_LEN = 64

    experiment_results = pd.DataFrame(experiment_results)
    display(experiment_results)
    experiment_results.to_csv(f"{OUT_DIR}/experiment_results.csv", index=False)
else:
    print("Experiments skipped. Set RUN_CONTROLLED_EXPERIMENTS=True to run them.")

## 11. Error analysis

Inspect the largest absolute errors from the fine-tuned model. Look for patterns such as abbreviations, reordered words, broad-versus-specific technical terms, and phrases that share vocabulary but differ in meaning. The table is based on the validation predictions produced above; use the examples to write a short interpretation in your report.

In [ ]:
# Keep both model and baseline predictions together for comparison.
val["abs_error"] = (val["pred"] - val["score"]).abs()
val["baseline_abs_error"] = (val["baseline_pred"] - val["score"]).abs()

error_columns = [
    "id", "context", "anchor", "target", "score", "pred",
    "abs_error", "baseline_pred", "baseline_abs_error"
]
worst_errors = val.sort_values("abs_error", ascending=False)[error_columns].head(20)
display(worst_errors)

# Summarize model performance by target score and CPC section.
val["cpc_section"] = val["context"].astype(str).str[0]
score_error_summary = val.groupby("score")["abs_error"].agg(["count", "mean", "median"]).reset_index()
section_error_summary = val.groupby("cpc_section")["abs_error"].agg(["count", "mean"]).reset_index()
print("Absolute error by true score:")
display(score_error_summary)
print("Absolute error by CPC section:")
display(section_error_summary)

# Save a single file containing the predictions and diagnostic columns.
val.to_csv(f"{OUT_DIR}/val_predictions_with_errors.csv", index=False)

### Error-analysis write-up

After reviewing the displayed examples, record observations supported by those rows:

1. Which types of phrase pairs produced the largest errors?
2. Did the model do better on exact/near-exact wording or on paraphrases?
3. Did any CPC section or score range appear more difficult?
4. Give one concrete example and explain why the prediction may have been wrong.

Do not claim a pattern unless it is visible in the validation examples or summary tables.

## 12. Submission

The competition expects `/kaggle/working/submission.csv` with columns `id,score` and one row per test row. Clipping to [0, 1] keeps the scores in the label range; it does not change the validation Pearson correlation much.

In [ ]:
test_pred = trainer.predict(ds_test).predictions.reshape(-1)
submission = pd.DataFrame({"id": test["id"], "score": np.clip(test_pred, 0, 1)})
assert len(submission) == len(test) and submission["score"].notna().all()
submission.to_csv(f"{OUT_DIR}/submission.csv", index=False)
submission.head()

## 13. Final report and submission notes

The notebook now contains a lexical baseline, two controlled experiment recipes, and validation error-analysis outputs. Run the controlled experiments when compute is available, then fill in the short written analysis using the results actually produced.

Record the following after the notebook has run:

- Fine-tuned model validation Pearson: use the value printed in Section 8.
- TF-IDF baseline validation Pearson: use the value printed in Section 9.
- Experiment A and B validation Pearson: use the values printed in Section 10.
- Public and private leaderboard scores: copy these from the competition page after submission.

The visible test file in a local notebook may contain only a few rows; the competition evaluation substitutes the hidden test set. The notebook writes `submission.csv` to `OUT_DIR`. Check that it has exactly the columns `id,score`, one row per test example, and no missing scores before submitting.

Do not invent leaderboard scores or experiment outcomes. If you skip the extra training runs, state that they were not run rather than reporting guessed values.